# Coastal SPM: final workflow

We estimate suspended particulate matter (SPM, mg/L) from the ratio of red to green satellite reflectance. Separate relationships are trained for Sentinel-2 and Sentinel-3 using field measurements from the Belgian coast and neighbouring French and Dutch waters.

We developed this method through experiments with SPM and Secchi depth. We compared bands, indices, simple regressions, random forests and support-vector regression using validation that keeps observations from the same acquisition date together. GLORIA spectral pretraining did not improve the S2-SPM neural network's reserved-date results. Wind, extra GLORIA satellite matchups and S3 centre pixels gave no consistent improvement in the models tested. We chose the two-coefficient red/green power law for its lower development log-error and simplicity. Random forests gave smaller errors in physical units in some comparisons.

Secchi had fewer usable matchups, although some random-forest results were promising. We focused the final application on SPM to compare one target across both sensors. The choice reflects these exploratory results and the project's scope. It does not establish that power laws are generally best or that Secchi cannot be retrieved.

The notebook covers image inspection, field–satellite pairing, model training, evaluation on excluded dates and regional mapping. It explains the chosen method and how we produced the maps.


## A2. Satellite image exploration

We inspect the optical inputs before estimating SPM. Sentinel-2 (S2) has 20 m output pixels and Sentinel-3 (S3) has nominal 300 m pixels. Remote-sensing reflectance, $R_{rs}$ (sr⁻¹), describes water-leaving radiance relative to the downwelling light reaching the water. Atmospheric correction separates the water signal from atmospheric contributions. Section 2 describes the products and quality checks.

The figures show green, red and red-edge bands, red/green = B4/B3, and the Normalized Difference Turbidity Index, NDTI = (B4 − B3)/(B4 + B3). Both index inputs must exceed 10⁻⁶ sr⁻¹ to avoid unstable ratios near zero. NDTI gives the same ordering as red/green, so the two indices are not independent evidence. The false-colour image assigns red-edge/red/green to RGB, with a fixed 0–0.03 sr⁻¹ range and square-root stretch.

Band and index close-ups cover 30 × 27 km. Wider comparisons cover 99 × 85 km. Maps are north-up in UTM 31N (EPSG:32631), with grey for masked pixels and nominal wavelength labels. Plot limits affect colours only. We do not smooth the data or blend tile boundaries. The two dates show individual conditions, not seasonal variation. The cell exports 300 dpi PNGs and SVGs to `figures/A2`, including individual panels.


In [ ]:
# A2: large band, index, optical and two-date figures, no new downloads
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

try:
    from google.colab import drive
except ImportError:
    A2_ROOT = Path.cwd()
    if not (A2_ROOT/'mapping_01/scenes.csv').is_file():
        A2_ROOT = A2_ROOT/'outputs/coastal_project'
else:
    drive.mount('/content/drive')
    A2_ROOT = Path('/content/drive/MyDrive/coastal_project')
A2_OUT = A2_ROOT/'figures/A2'
A2_OUT.mkdir(parents=True, exist_ok=True)
a2_scenes = pd.read_csv(A2_ROOT/'mapping_01/scenes.csv').query('region == "Belgium"')
a2_data, a2_labels = {}, {}
for scene in a2_scenes.itertuples():
    path = A2_ROOT/scene.asset
    assert hashlib.sha256(path.read_bytes()).hexdigest() == scene.asset_sha256
    key = scene.sensor+'_'+scene.date[:4]
    with np.load(path) as image:
        a2_data[key] = {b: image[b] for b in ['x','y','valid','green','red','red_edge']}
    a2_labels[key] = f'{scene.sensor} · {scene.date} · {scene.pixel_size_m} m'

def a2_plot(name, panels, view, show=True, images=a2_data, labels=a2_labels, out=A2_OUT, columns=None):
    columns = columns or len(panels)
    rows = (len(panels)+columns-1)//columns
    height = 7.8 if rows == 1 else 7.8*(view[3]-view[2])/(view[1]-view[0])+.45
    fig, axes = plt.subplots(rows, columns, figsize=(7.8*columns, height*rows), layout='constrained')
    for ax, (key, values, title, limits, cmap, unit) in zip(np.asarray(axes).ravel(), panels):
        data = images[key]
        x, y = data['x']/1000, data['y']/1000
        style = dict(vmin=limits[0], vmax=limits[1], cmap=cmap) if limits else {}
        ax.set_facecolor('#e8e8e8')
        if x.ndim == 1:
            im = ax.imshow(values, extent=[x[0]-.01,x[-1]+.01,y[-1]-.01,y[0]+.01],
                           origin='upper', interpolation='nearest', **style)
        else:
            im = ax.pcolormesh(x, y, values, shading='nearest', rasterized=True, **style)
        ax.set(xlim=view[:2], ylim=view[2:], aspect='equal', title=title)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.text(.015,.02,labels[key],transform=ax.transAxes,fontsize=8,
                bbox=dict(facecolor='white',edgecolor='none',alpha=.8,pad=2))
        length = 5 if view[1]-view[0] < 50 else 20
        right, bottom = view[1]-1.5, view[2]+1.2
        ax.plot([right-length,right],[bottom,bottom],color='black',lw=1.2,
                path_effects=[pe.withStroke(linewidth=2.5,foreground='white')])
        ax.text(right-length/2,bottom+.5,f'{length} km',color='black',ha='center',fontsize=8,
                path_effects=[pe.withStroke(linewidth=1.2,foreground='white')])
        if limits:
            extend = 'max' if unit == 'Estimated SPM (mg/L)' else 'both'
            cb = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.025, pad=.018, aspect=55, extend=extend)
            cb.set_label(unit, fontsize=9)
            cb.ax.tick_params(labelsize=8, length=2)
    for suffix in ['png','svg']:
        fig.savefig(out/(name+'.'+suffix), dpi=300, bbox_inches='tight', pad_inches=.04)
    if show:
        plt.show()
    return fig

detail, overview = (495,525,5685,5712), (443,542,5655,5740)  # UTM kilometres
s2 = a2_data['S2_2019']
a2_band_panels = [
    ('S2_2019',s2[b],title,(0,.03),'viridis','Rrs (sr⁻¹)')
    for b,title in [('green','Green · B3, 560 nm'),('red','Red · B4, 665 nm'),('red_edge','Red edge · B5, 704 nm')]
]
a2_bands = a2_plot('01_bands_detail', a2_band_panels, detail)
usable = s2['valid'] & (s2['red']>1e-6) & (s2['green']>1e-6)
a2_ratio = np.where(usable, s2['red']/s2['green'], np.nan)
a2_ndti = np.where(usable, (s2['red']-s2['green'])/(s2['red']+s2['green']), np.nan)
a2_index_panels = [
    ('S2_2019',a2_ratio,'Red / green',(0,1),'viridis','Ratio (dimensionless)'),
    ('S2_2019',a2_ndti,'NDTI',(-.8,.2),'magma','NDTI (dimensionless)')
]
a2_indices = a2_plot('02_indices_detail', a2_index_panels, detail)
a2_rgb = {}
for key in ['S2_2019','S3_2019']:
    image = a2_data[key]
    rgb = np.sqrt(np.clip(np.stack([image[b] for b in ['red_edge','red','green']],axis=-1)/.03,0,1))
    a2_rgb[key] = np.dstack([np.nan_to_num(rgb),image['valid']])
a2_optical = a2_plot('03_false_colour_overview', [
    (key,a2_rgb[key],label,None,None,'') for key,label in [('S2_2019','Sentinel-2 · red-edge/red/green'),('S3_2019','Sentinel-3 · red-edge/red/green')]
], overview)
a2_time = a2_plot('04_ratio_two_dates', [
    (key,a2_data[key]['red']/a2_data[key]['green'],date,(0,1),'viridis','Red / green (dimensionless)')
    for key,date in [('S3_2019','23 July 2019'),('S3_2020','19 May 2020')]
], overview)
panel_names = ['B3_green','B4_red','B5_red_edge','red_green','NDTI']
for name, panel in zip(panel_names, a2_band_panels+a2_index_panels):
    plt.close(a2_plot('panel_'+name, [panel], detail, show=False))


## 1. Method (A5, B1)

We pair each field SPM measurement with nearby satellite reflectance. Red reflectance ($R$) divided by green reflectance ($G$) gives the input $x$. A power law converts it into predicted SPM:

$$x=\frac{R}{G},\qquad \widehat{\mathrm{SPM}}=A x^b$$

$A$ sets the scale and $b$ controls how strongly SPM changes with the ratio. If $b=2$, doubling red/green multiplies predicted SPM by four. We estimate both coefficients separately for each sensor from the regional observations. The relationship may differ in other water types.

| Sensor | Green, $G$ | Red, $R$ | Model input |
| --- | --- | --- | --- |
| Sentinel-2 MultiSpectral Instrument (MSI) | B3, about 560 nm | B4, about 665 nm | B4/B3 |
| Sentinel-3 Ocean and Land Colour Instrument (OLCI) | Oa06, 560 nm | Oa08, 665 nm | Oa08/Oa06 |

To calculate $A$ and $b$, we take base-10 logarithms of the positive ratio and measured SPM. This turns the power law into a straight line:

$$\log_{10}(\mathrm{SPM})=a+b\log_{10}(x),\qquad A=10^a$$

Ordinary least squares minimises squared differences between predicted and measured log-SPM, giving each observation equal weight. This emphasises proportional errors: predicting twice or half the measured value gives the same squared log-error. We convert predictions directly back to mg/L without correcting for retransformation bias. They therefore need not estimate arithmetic mean SPM.

[Knaeps et al. (2010), Section 5–6](https://doi.org/10.3390/rs2092040) used exponential band-ratio regressions for SPM at different wavelengths. [Alikas & Kratzer (2017), Eq. 1](https://doi.org/10.1016/j.ecolind.2017.02.007) fitted a band-ratio power law for Secchi depth. We used these studies to guide the regression approach, then fitted our own red/green SPM relationship for the available sensors and regional data.

The development comparisons favoured red/green power laws over random forests (RF) on the primary log-error measure. RF had slightly lower physical-unit RMSE for S2, but the power law performed better for S3 and was simpler to apply. We chose it for both sensors. The model uses no wind, pretraining, GLORIA training observations or fitted input scaling. Red-edge reflectance is used for quality checks and optical figures, not prediction.


## 2. Open the prepared data (A1, B1)

Field data come from [LifeWatch](https://doi.org/10.14284/652), [Rijkswaterstaat](https://rijkswaterstaatdata.nl/waterdata/) and [SOMLIT-Wimereux](https://doi.org/10.17882/96639). We use positive SPM measurements at known near-surface sampling depths of 0–5 m. Accepted quality classes are LifeWatch Good/Probably good data, RWS codes 00/90 with checked or definitive status, and SOMLIT codes 2/6. We exclude missing, below-detection and rejected values. Compatible accepted replicates from the same source, station, timestamp, depth and method are averaged.

Usable pairs span November 2015–August 2026 for S2 and May 2016–October 2025 for S3, with 17 station identifiers per sensor. Sampling, satellite availability and quality screening produce irregular coverage rather than continuous time series.

We corrected S2 Level-1C imagery with ACOLITE 20260421.0 using dark spectrum fitting, residual sun-glint correction and 20 m output pixels, without ancillary-data retrieval. Pixels must have `l2_flags = 0` and finite green, red and red-edge reflectance. S3 uses Copernicus Level-2 Water Full Resolution (WFR), Collection 003. We divide its directional water reflectance by $\pi$ to obtain $R_{rs}$. The correction chains differ, so we calibrate each sensor separately. [ACOLITE implementation](https://github.com/acolite/acolite/blob/20260421.0/acolite/acolite/acolite_l2w.py).

For S3, the [Collection 003 product notice, Table 1](https://user.eumetsat.int/s3/eup-strapi-media/S3_PN_OLCI_L2_M_003_02_Sentinel_3_Product_Notice_OLCI_Level_2_Ocean_Colour_new_596245cb3a.pdf) defines the water-reflectance mask. We require WATER or INLAND_WATER and exclude CLOUD, CLOUD_AMBIGUOUS, CLOUD_MARGIN, INVALID, COSMETIC, SATURATED, SUSPECT, HISOLZEN, HIGHGLINT, SNOW_ICE, AC_FAIL, WHITECAPS, ADJAC and RWNEG_O2–O8. Flag bits are read from each product's metadata.

Satellite and field times must differ by no more than three hours. We centre windows on the nearest pixel, within 15 m for S2 or 750 m for S3. S2 uses 5 × 5 pixels at 20 m (100 × 100 m), with at least 20/25 valid. S3 uses 3 × 3 pixels at nominal 300 m (about 900 × 900 m), with 9/9 valid. The three core bands use the same valid pixels. We chose these window sizes and time limits for this project. Overlapping images are ranked by verified quality, coverage, time difference and product ID. We select one pair per sensor, collection and field sample without using SPM values. We inspected variation within windows but did not use it to reject observations.

We follow LifeWatch's mg/L unit field, although its parameter label gives conflicting information. SOMLIT times are assumed to use Europe/Paris, including daylight saving, because the provider's convention is unconfirmed. We use the recorded offsets for other sources, including Dutch fixed +01:00 timestamps. These uncertainties could affect concentrations and image matching.

This notebook reads prepared pairs and regional rasters, so it does not repeat downloads or atmospheric correction. Run it from this folder or the project workspace. For Colab, place the extracted `coastal_project` folder in My Drive and adjust the path below if needed. Analysis requires NumPy, pandas and Matplotlib. File checks verify the input versions.


In [ ]:
# B1: local or Google Drive paths
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    PROJECT = Path.cwd()
    if not (PROJECT / 'data/satellite_audit/analysis_matchups.csv').is_file():
        PROJECT = PROJECT / 'outputs/coastal_project'
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/coastal_project')

assert (PROJECT / 'data/satellite_audit/analysis_matchups.csv').is_file(), 'Check PROJECT.'
print(f'Python {sys.version.split()[0]}, NumPy {np.__version__}, pandas {pd.__version__}')


In [ ]:
# B1-C1: fixed choices, no model or feature search
BANDS = {'S2': ('B3', 'B4'), 'S3': ('Oa06', 'Oa08')}  # green, red
MIN_RRS = 1e-6  # sr^-1, existing index safeguard, not a detection limit
PRIMARY_METRIC = 'log10_RMSE'
SEED = 42
N_BOOT = 1000
CI_LEVEL = 0.95


In [ ]:
# B1: verify the exact prepared inputs and saved experiment records
INPUT_SHA256 = {
    "data/satellite_audit/analysis_matchups.csv": "a01d5e4c7f79b846887a098541bec2fec776e17bb998b9965d8a94e421698961",
    "data/model_preparation/local_assignments.csv": "f88a296ad791b362f516570e6e6faaf9f78f161ba165b5ae93894289e3d36058",
    "models/local_baselines_01/fold_assignments.csv": "df4d6672d19a598fd12a1c5a87477d356aa2b73aad23042218343b5aeaf69e5a",
    "models/statistical_fits_01/development_results.csv": "37af31d1ef61fede728b3081bf5db77fbc21aa7eb8da83b8a87b6d4270a110e0",
    "models/S2_SPM/test_01/test_predictions.csv": "82f75a4c202f26be4704e5d5515eaf38ee339421afecd1f0cd1dd8c2d214358c"
}
for name, expected in INPUT_SHA256.items():
    assert hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() == expected, name
print('Five input snapshots verified.')


In [ ]:
# B1: read previous development results, do not repeat model selection
previous = pd.read_csv(PROJECT / 'models/statistical_fits_01/development_results.csv')
previous.loc[previous['target'].eq('SPM') & previous['model'].isin(
    ['Power: red/green', 'Random forest']),
    ['sensor', 'model', 'n', 'dates', 'log10_RMSE', 'RMSE']].round(3)


## 3. Check the sample lists (B1, C1)

Each row pairs one field SPM sample with one satellite acquisition for a given sensor. Several stations can share an image, and a sample can have both an S2 and an S3 pair. Row counts therefore differ from counts of dates or independent field measurements.

We average red and green reflectance over the same valid window pixels, then divide the means. We do not average individual pixel ratios. Both means must exceed 10⁻⁶ sr⁻¹, a safeguard against unstable ratios near zero rather than a detection limit. Pixel values that pass the source mask are used as recorded, including negatives. Missing ratios are not filled.

Saved assignments divide observations into training (`development`) and reserved `test` sets. We randomly reserved 20% of available acquisition dates, rounded up, using seed 42 and shared assignments across sensors and targets. The usable proportions differ by sensor after input selection. The other dates form three groups for Section 4. The code checks that training and test sets have no shared sample IDs, acquisition dates or product IDs.


In [ ]:
# B1: join the original observations and fixed assignments
matchups = pd.read_csv(PROJECT / 'data/satellite_audit/analysis_matchups.csv',
                       dtype={'collection': str, 'station_id': str})
roles = pd.read_csv(PROJECT / 'data/model_preparation/local_assignments.csv',
                   dtype={'collection': str})
folds = pd.read_csv(PROJECT / 'models/local_baselines_01/fold_assignments.csv')
assert matchups['row_id'].is_unique and roles['row_id'].is_unique
assert roles.groupby('acquisition_date')['role'].nunique().eq(1).all()
assert roles.groupby('sample_id')['role'].nunique().eq(1).all()

spm = matchups.loc[matchups['target'].eq('SPM') &
    (matchups['sensor'].eq('S2') | matchups['collection'].eq('003'))].copy()
spm = spm.merge(roles[['row_id', 'role']], on='row_id', how='left', validate='one_to_one')
spm = spm.merge(folds[['row_id', 'fold', 'input_complete']],
                on='row_id', how='left', validate='one_to_one')
spm['fold'] = spm['fold'].astype('Int64')
assert spm['primary_verified'].all() and spm['role'].isin(['development', 'test']).all()
assert spm['target_unit'].eq('mg/L').all() and spm['target_value'].gt(0).all()
assert not spm['source'].str.contains('GLORIA', case=False).any()


In [ ]:
# B1: the final model needs red/green, not all eight earlier inputs
spm['red_green'] = np.nan
for sensor, (green_band, red_band) in BANDS.items():
    use = spm['sensor'].eq(sensor)
    green = spm.loc[use, 'raw__Rrs_' + green_band]
    red = spm.loc[use, 'raw__Rrs_' + red_band]
    spm.loc[use, 'red_green'] = (red / green).where(
        green.gt(MIN_RRS) & red.gt(MIN_RRS))
np.testing.assert_allclose(spm['red_green'], spm['raw__red_green_ratio'],
                           rtol=1e-10, atol=1e-12, equal_nan=True)
spm['input_ready'] = np.isfinite(spm['red_green']) & spm['red_green'].gt(0)

development = spm.loc[spm['role'].eq('development') & spm['input_ready']].copy()
evaluation = spm.loc[spm['role'].eq('test') & spm['input_ready']].copy()
for column in ['row_id', 'sample_id', 'acquisition_date', 'product_id']:
    assert set(development[column]).isdisjoint(evaluation[column])
assert development['fold'].notna().all()
assert development.groupby(['sensor', 'acquisition_date'])['fold'].nunique().eq(1).all()


In [ ]:
# B1-C1: report coverage before any fitting
counts = spm.groupby(['sensor', 'role']).agg(
    prepared=('row_id', 'size'), prepared_dates=('acquisition_date', 'nunique'),
    usable=('input_ready', 'sum'))
ready_counts = spm.loc[spm['input_ready']].groupby(['sensor', 'role']).agg(
    usable_dates=('acquisition_date', 'nunique'), stations=('station_id', 'nunique'))
counts = counts.join(ready_counts)
counts['unusable'] = counts['prepared'] - counts['usable']
counts


S2 has 63 training observations from 37 dates and 17 test observations from 9 dates. S3 has 106 training observations from 62 dates and 27 test observations from 15 dates. One prepared S3 training row has no usable ratio. The tables show sample and station coverage for each set.

The earlier eight-input comparison used 104 S3 training rows. Two more rows have valid red/green but a missing red-edge index, so they can be used by the final model. They use the same quality masks and date assignments. The earlier scores describe 104 rows. We made this input-completeness decision without using test errors.


In [ ]:
# B1: verify the saved development groups, including the two restored rows
fold_counts = development.groupby(['sensor', 'fold']).agg(
    observations=('row_id', 'size'), dates=('acquisition_date', 'nunique'))
fold_counts


## 4. Training and checks across dates (B1, C1, C2)

We test predictions for other dates within the sampled region. Observations from the same acquisition date are grouped because they can share imagery, atmospheric conditions and processing errors, following [Roberts et al. (2017)](https://doi.org/10.1111/ecog.02881). Grouping prevents same-date overlap between training and evaluation. It does not guarantee that different dates are independent, or test new coastlines and future years.

The first check uses only training observations:

1. Learn $A$ and $b$ from two of the three saved date groups, then predict SPM for the third.
2. Repeat with each group left out once. Every training observation receives one prediction from a model that did not use its date.
3. Compare these predictions with the field measurements, pooling their errors with equal weight per observation.

This is three-fold date-grouped cross-validation, following [scikit-learn's grouped procedure](https://scikit-learn.org/1.7/modules/cross_validation.html#group-k-fold). We chose three groups and an approximately 20% test reservation. After these checks, we estimate the final $A$ and $b$ from all training observations and evaluate them on the reserved dates. Those measurements are excluded from coefficient estimation, following [data-leakage guidance](https://scikit-learn.org/1.7/common_pitfalls.html#data-leakage). We do not change settings or inputs after seeing the final scores.

The training dates were used earlier to choose the model, so these checks are development results. The test history also needs qualification: S2 test results were viewed during the neural-network experiments, eight S3 test measurements were in that S2 test set, and earlier correlations included test observations. The final fits exclude these observations, but the evaluation is not a fully unseen test.

To estimate uncertainty in test errors, we repeatedly sample whole dates with replacement and include all station observations from each selected date. A date can appear several times or not at all. We recalculate errors for 1,000 samples per sensor, seed 42, and use the 2.5th–97.5th percentiles as approximate 95% intervals. This [cluster bootstrap](https://doi.org/10.1111/j.1467-9868.2007.00593.x) assumes independent dates. We do not retrain the models. The intervals describe uncertainty in the error measures, not individual pixels or atmospheric correction. They are based on only 9 S2 and 15 S3 test dates.


In [ ]:
# C1: count earlier test-label exposure without reading its predictions
previous_test = pd.read_csv(PROJECT / 'models/S2_SPM/test_01/test_predictions.csv',
                            usecols=['sample_id'])
evaluation['field_label_seen_in_S2_test'] = evaluation['sample_id'].isin(previous_test['sample_id'])
evaluation.groupby('sensor').agg(
    observations=('row_id', 'size'), dates=('acquisition_date', 'nunique'),
    labels_in_previous_S2_test=('field_label_seen_in_S2_test', 'sum'))


In [ ]:
# B1: estimate the intercept and slope of a straight line in log space
def fit_power_law(rows):
    x = np.log10(rows['red_green'].to_numpy())
    y = np.log10(rows['target_value'].to_numpy())
    a, b = np.polynomial.polynomial.polyfit(x, y, 1)
    return float(a), float(b)


In [ ]:
# B1-C1: train on two saved date groups and predict the third
prediction_rows, round_coefficients = [], []
for sensor, data in development.groupby('sensor'):
    for group in sorted(data['fold'].unique()):
        train = data.loc[data['fold'].ne(group)]
        check = data.loc[data['fold'].eq(group)]
        assert set(train['acquisition_date']).isdisjoint(check['acquisition_date'])
        a, b = fit_power_law(train)
        predicted = 10.0 ** (a + b * np.log10(check['red_green'].to_numpy()))
        assert np.isfinite(predicted).all() and (predicted > 0).all()
        part = check[['row_id', 'sample_id', 'sensor', 'source', 'station_id',
                      'acquisition_date', 'fold', 'target_value', 'red_green']].copy()
        part['predicted_spm_mg_l'] = predicted
        part['outside_training_ratio_range'] = ~check['red_green'].between(
            train['red_green'].min(), train['red_green'].max())
        prediction_rows.append(part)
        round_coefficients.append({'sensor': sensor, 'fold': int(group),
            'n_train': len(train), 'n_check': len(check), 'a': a, 'b': b,
            'ratio_min': train['red_green'].min(), 'ratio_max': train['red_green'].max()})

validation_predictions = pd.concat(prediction_rows, ignore_index=True)
assert validation_predictions['row_id'].is_unique
assert set(validation_predictions['row_id']) == set(development['row_id'])
assert set(validation_predictions['row_id']).isdisjoint(evaluation['row_id'])
round_coefficients = pd.DataFrame(round_coefficients)
print(f'{len(validation_predictions)} predictions, each excluding its observation date from training.')


In [ ]:
# C1: errors from predictions for dates excluded during training
def prediction_errors(observed, predicted):
    error = predicted - observed
    total_variation = np.sum((observed - observed.mean())**2)
    return {'MAE': np.mean(abs(error)), 'RMSE': np.sqrt(np.mean(error**2)),
            'bias': np.mean(error),
            'R2': 1 - np.sum(error**2) / total_variation if total_variation > 0 else np.nan,
            'log10_RMSE': np.sqrt(np.mean((np.log10(predicted) - np.log10(observed))**2))}

scores = []
for sensor, data in validation_predictions.groupby('sensor'):
    scores.append({'sensor': sensor, 'n': len(data),
        'dates': data['acquisition_date'].nunique(),
        **prediction_errors(data['target_value'].to_numpy(), data['predicted_spm_mg_l'].to_numpy())})
validation_results = pd.DataFrame(scores)
validation_results.set_index('sensor').round(3)


In [ ]:
# B1: train the final two curves using all training observations, no test predictions
parameters = []
for sensor, data in development.groupby('sensor'):
    a, b = fit_power_law(data)
    parameters.append({'sensor': sensor, 'n': len(data),
        'dates': data['acquisition_date'].nunique(), 'a': a, 'b': b, 'multiplier': 10.0**a,
        'ratio_min': data['red_green'].min(), 'ratio_max': data['red_green'].max()})
parameters = pd.DataFrame(parameters)
parameters.set_index('sensor').round(6)


For dates excluded during the training checks, RMSE is 34.27 mg/L for S2 and 15.82 mg/L for S3. Bias is −6.64 and −3.71 mg/L, so both models underestimate SPM on average. The sensors were evaluated on different observations, which prevents attributing the difference to the sensors alone.

Using all training observations gives the final equations, with predicted SPM in mg/L:

$$\text{Sentinel-2:}\quad \widehat{\mathrm{SPM}}=143.46\left(\frac{B4}{B3}\right)^{2.864}$$

$$\text{Sentinel-3:}\quad \widehat{\mathrm{SPM}}=102.81\left(\frac{Oa08}{Oa06}\right)^{1.731}$$

The displayed equations are rounded. Evaluation and mapping use full-precision coefficients, with `multiplier` equal to $A=10^a$. The saved training ratio ranges identify later inputs outside calibration. The next cells read and verify these values. [Training records](models/final_spm_01/run.json).


### Evaluation against reserved measurements

We apply the final equations to all 17 reserved S2 observations and 27 reserved S3 observations, then compare predictions with field SPM. These results do not trigger further model changes.

| Measure | Interpretation | Units |
| --- | --- | --- |
| MAE | Average absolute error | mg/L |
| RMSE | Gives more weight to large errors | mg/L |
| Bias | Average prediction minus measurement, negative means underestimation | mg/L |
| R² | Error relative to predicting the test-set mean, 1 is perfect and negative is worse than that mean | None |
| log10-RMSE | Error on the log scale, describing proportional disagreement | None |

Lower MAE, RMSE and log10-RMSE indicate smaller errors, while bias should be close to zero. We used log10-RMSE for model selection and report physical-unit errors to show the size of concentration errors. Definitions follow [scikit-learn](https://scikit-learn.org/1.7/modules/model_evaluation.html#regression-metrics). R² is undefined for bootstrap draws with identical measurements. Those draws are excluded from its interval only, with their number reported.


In [ ]:
# C1: use the saved curves, without recalculating their coefficients
coefficient_file = PROJECT / 'models/final_spm_01/coefficients.csv'
assert hashlib.sha256(coefficient_file.read_bytes()).hexdigest() == (
    '75ec4784a697879acd67e144e7b1a6d80ccf771d1c20852ce5b8ea99d449b31d')
frozen_parameters = pd.read_csv(coefficient_file)
assert frozen_parameters['sensor'].is_unique and set(frozen_parameters['sensor']) == {'S2', 'S3'}
test_predictions = evaluation[['row_id', 'sample_id', 'sensor', 'source', 'station_id',
    'acquisition_date', 'datetime_utc', 'season', 'target_value', 'target_unit', 'red_green',
    'field_label_seen_in_S2_test']].copy()
test_predictions['predicted_spm_mg_l'] = np.nan
test_predictions['outside_training_ratio_range'] = False
for model in frozen_parameters.itertuples(index=False):
    use = test_predictions['sensor'].eq(model.sensor)
    ratio = test_predictions.loc[use, 'red_green']
    test_predictions.loc[use, 'predicted_spm_mg_l'] = 10.0 ** (model.a + model.b * np.log10(ratio))
    test_predictions.loc[use, 'outside_training_ratio_range'] = ~ratio.between(
        model.ratio_min, model.ratio_max)
assert test_predictions['row_id'].is_unique and set(test_predictions['row_id']) == set(evaluation['row_id'])
assert np.isfinite(test_predictions['predicted_spm_mg_l']).all()
assert test_predictions['predicted_spm_mg_l'].gt(0).all()


In [ ]:
# C1: evaluate every reserved observation, with no model changes
rows = []
for sensor, data in test_predictions.groupby('sensor'):
    rows.append({'sensor': sensor, 'n': len(data), 'dates': data['acquisition_date'].nunique(),
        **prediction_errors(data['target_value'].to_numpy(), data['predicted_spm_mg_l'].to_numpy())})
test_results = pd.DataFrame(rows)
test_results.set_index('sensor').round(3)


In [ ]:
# C1-C2: resample whole dates, keeping predictions fixed
bootstrap_rows, interval_rows = [], []
tail = (1 - CI_LEVEL) / 2
for sensor, data in test_predictions.groupby('sensor'):
    data = data.reset_index(drop=True)
    date_rows = [group.index.to_numpy() for _, group in data.groupby('acquisition_date', sort=True)]
    rng = np.random.default_rng(SEED)
    draws = []
    for draw in range(N_BOOT):
        selected = rng.integers(0, len(date_rows), len(date_rows))
        sample = data.loc[np.concatenate([date_rows[i] for i in selected])]
        draws.append({'sensor': sensor, 'draw': draw, 'sample_n': len(sample),
            **prediction_errors(sample['target_value'].to_numpy(), sample['predicted_spm_mg_l'].to_numpy())})
    draws = pd.DataFrame(draws)
    bootstrap_rows.append(draws)
    for metric in ['MAE', 'RMSE', 'bias', 'R2', 'log10_RMSE']:
        values = draws.loc[np.isfinite(draws[metric]), metric]
        lower, upper = np.quantile(values, [tail, 1 - tail]) if len(values) else (np.nan, np.nan)
        interval_rows.append({'sensor': sensor, 'metric': metric,
            'estimate': test_results.set_index('sensor').loc[sensor, metric],
            'lower_95': lower, 'upper_95': upper, 'valid_draws': len(values),
            'undefined_draws': N_BOOT - len(values)})
bootstrap_results = pd.concat(bootstrap_rows, ignore_index=True)
test_intervals = pd.DataFrame(interval_rows)
test_intervals.set_index(['sensor', 'metric']).round(3)


In [ ]:
# C1: observed and predicted SPM on the same scales, all observations shown
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8.4, 4), sharex=True, sharey=True, layout='constrained')
upper = test_predictions[['target_value', 'predicted_spm_mg_l']].to_numpy().max() * 1.05
for ax, (sensor, data), colour in zip(axes, test_predictions.groupby('sensor'), ['#167d8d', '#cf713d']):
    ax.scatter(data['target_value'], data['predicted_spm_mg_l'], s=34, color=colour, alpha=0.85)
    ax.plot([0, upper], [0, upper], '--', color='0.45', linewidth=1, label='Perfect agreement')
    ax.set(title=f'Sentinel-{sensor[-1]}: n={len(data)}, {data.acquisition_date.nunique()} dates',
           xlabel='Observed SPM (mg/L)', xlim=(0, upper), ylim=(0, upper))
    ax.set_aspect('equal', adjustable='box')
axes[0].set_ylabel('Predicted SPM (mg/L)')
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
plt.show()


On the reserved dates, RMSE is 35.47 mg/L for S2 and 43.47 mg/L for S3, with R² of 0.107 and 0.102. Both models perform poorly at high concentrations. S2 predicts 36.0 mg/L for a measured 142 mg/L, and S3 predicts 24.2 mg/L for 150 mg/L. All test ratios are inside the training ranges, so checking the input range would not catch these errors.

The approximate 95% bootstrap RMSE intervals are 11.10–51.19 mg/L for S2 and 8.58–63.78 mg/L for S3. The limited number of test dates gives wide intervals. Different observation sets and shared field measurements also prevent a controlled sensor comparison. We use the same equations for mapping, with these accuracy limits.

In the figure, the dashed line marks perfect agreement and points below it are underestimates. All predictions are shown, including the largest errors. [Predictions](models/final_spm_test_01/test_predictions.csv), [errors](models/final_spm_test_01/test_results.csv), [intervals](models/final_spm_test_01/test_intervals.csv).

This evaluates predictions from station-window means. The next section examines mapped patterns in Belgium and another region. A plausible-looking map does not establish accurate concentrations.


We group the same reserved predictions by station and season, without retraining. `n` counts observations and `dates` counts acquisition dates. MAE and bias are in mg/L, with negative bias indicating underestimation. Seasons follow the field timestamp: DJF = Dec–Feb, MAM = Mar–May, JJA = Jun–Aug and SON = Sep–Nov. Groups with no observations have no error estimate. Each dot below is one prediction minus its measurement. The small groups do not support reliable station rankings.


In [ ]:
# C1: describe station errors using the existing reserved predictions
group_errors = test_predictions.copy()
group_errors['error_mg_l'] = group_errors['predicted_spm_mg_l'] - group_errors['target_value']
group_errors['absolute_error_mg_l'] = group_errors['error_mg_l'].abs()
station_errors = group_errors.groupby(['sensor', 'station_id']).agg(
    n=('row_id', 'size'), dates=('acquisition_date', 'nunique'),
    MAE_mg_l=('absolute_error_mg_l', 'mean'), bias_mg_l=('error_mg_l', 'mean'))
station_errors.round(2)


In [ ]:
# C1: include empty seasons without assigning them zero error
season_order = ['DJF', 'MAM', 'JJA', 'SON']
season_errors = group_errors.groupby(['sensor', 'season']).agg(
    n=('row_id', 'size'), dates=('acquisition_date', 'nunique'),
    MAE_mg_l=('absolute_error_mg_l', 'mean'), bias_mg_l=('error_mg_l', 'mean'))
season_errors = season_errors.reindex(pd.MultiIndex.from_product(
    [['S2', 'S3'], season_order], names=['sensor', 'season']))
season_errors[['n', 'dates']] = season_errors[['n', 'dates']].fillna(0).astype(int)
season_errors.round(2).fillna('n/a')


In [ ]:
# C1: show every individual error, without fitting or filtering
fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True, layout='constrained')
limit = group_errors['error_mg_l'].abs().max() * 1.08
for i, (sensor, colour) in enumerate([('S2', '#167d8d'), ('S3', '#cf713d')]):
    part = group_errors.loc[group_errors['sensor'].eq(sensor)]
    for ax, column in zip(axes[i], ['station_id', 'season']):
        order = sorted(part[column].unique()) if column == 'station_id' else season_order
        positions = {name: j for j, name in enumerate(order)}
        counts = part[column].value_counts()
        ax.scatter(part['error_mg_l'], part[column].map(positions),
                   s=32, color=colour, alpha=.8, edgecolors='none')
        ax.set_yticks(range(len(order)), [f'{name} (n={counts.get(name, 0)})' for name in order])
        ax.set_ylim(len(order) - .5, -.5)
        ax.set_xlim(-limit, limit)
        ax.axvline(0, color='0.45', linestyle='--', linewidth=1)
        ax.grid(axis='x', alpha=.2)
        ax.set_axisbelow(True)
        ax.tick_params(axis='y', labelsize=9)
        ax.set_title(f"Sentinel-{sensor[-1]}: {'station' if column == 'station_id' else 'season'}")
        ax.set_xlabel('Prediction minus measurement (mg/L)')

C1_OUT = PROJECT / 'figures/C1'
C1_OUT.mkdir(parents=True, exist_ok=True)
station_errors.to_csv(C1_OUT / 'errors_by_station.csv')
season_errors.to_csv(C1_OUT / 'errors_by_season.csv')
for suffix in ['png', 'svg']:
    fig.savefig(C1_OUT / f'reserved_errors_by_group.{suffix}', dpi=300)
plt.show()


S3 spring MAE is 45.39 mg/L, compared with 13.55 mg/L in summer. Spring includes its two highest SPM measurements, 144 and 150 mg/L, which are strongly underestimated. This does not establish a seasonal effect because concentrations, stations and dates differ between groups. There are only 1–3 observations per station for each sensor. S2 has no winter test data and S3 has only one winter observation, so winter performance is largely untested. No model settings or coefficients are changed after these checks.


## 5. Regional SPM maps (A2, B2–B4)

We calculate red/green for each valid image pixel and apply its sensor's equation from Section 4. Prediction does not require a field measurement. Inputs outside the training minimum–maximum range are flagged as extrapolations, but we do not clip or recalibrate their predictions. These maps use individual pixels, while training and evaluation used window means. We have therefore not measured individual-pixel accuracy.

We use the following scenes to compare spatial patterns, dates and application outside the calibration region:

| Area and date | Comparison |
| --- | --- |
| Belgium, 23 July 2019 | S2 and S3 on the same date |
| Belgium, 19 May 2020 | S3 compared with the Belgian 2019 image |
| French Guiana, 6 July 2017 | Both unchanged models applied outside their calibration region |

We chose dates from image availability and station quality coverage before examining SPM predictions. The Guiana locations came from GLORIA and were used in earlier exploration, but none enters the final calibration. The maps show geographic application, not an independent external accuracy test.

Acquisition and atmospheric correction were done outside this notebook. The regional rasters use the products and quality rules in Section 2. S2 tiles share an aligned 20 m UTM grid. In overlaps, we use the first valid pixel in a fixed tile order. S3 uses its native curved grid at nominal 300 m resolution. We apply no additional resampling or smoothing during mapping. S2 correction covered larger areas than the training extracts, which can affect atmospheric correction. [Acquisition settings and source products](mapping_01/plan.json).

We use regional subsets, not whole scenes. Paired S2/S3 acquisitions are about 57 minutes apart in Belgium and 50 minutes apart in Guiana. Differences between maps can therefore reflect acquisition time, resolution and processing as well as the sensor.


In [ ]:
# A2-B4: load verified regional rasters, no downloads or training
scenes = pd.read_csv(PROJECT / 'mapping_01/scenes.csv')
positions = pd.read_csv(PROJECT / 'mapping_01/sampling_positions.csv')
maps, map_rows = {}, []
for scene in scenes.itertuples(index=False):
    path = PROJECT / scene.asset
    assert hashlib.sha256(path.read_bytes()).hexdigest() == scene.asset_sha256
    with np.load(path) as source:
        data = {key: source[key] for key in source.files}
    model = frozen_parameters.set_index('sensor').loc[scene.sensor]
    ratio = data['red'].astype(float) / data['green'].astype(float)
    valid = data['valid']
    assert np.isfinite(ratio[valid]).all() and (ratio[valid] > 0).all()
    data['spm'] = np.where(valid, 10.0 ** (model.a + model.b * np.log10(ratio)), np.nan)
    data['outside'] = valid & ((ratio < model.ratio_min) | (ratio > model.ratio_max))
    maps[(scene.case, scene.sensor)] = data
    map_rows.append({'region': scene.region, 'date': scene.date, 'sensor': scene.sensor,
        'valid_pixels': int(valid.sum()), 'outside_training_percent': 100 * data['outside'].sum() / valid.sum(),
        'maximum_spm_mg_l': float(data['spm'][valid].max())})
map_summary = pd.DataFrame(map_rows)
print(f'{len(scenes)}/5 regional rasters ready.')
map_summary.round(2)


SPM maps share a 0–100 mg/L colour scale. Values above 100 use the highest colour without changing their numerical predictions. Grey marks masked pixels, land or missing coverage. Views are north-up, with 5 km scale bars and no spatial smoothing.

Belgian close-ups cover 30 × 27 km in UTM 31N (EPSG:32631). Guiana views cover 45 × 35 km in UTM 22N (EPSG:32622). Zooming affects the view only. The summary table describes the full regional rasters. Figures are exported as 300 dpi PNGs and SVGs to `figures/final_results`.

The table reports the percentage of valid pixels outside the training ratio range and the maximum prediction, including values above the colour scale. Flag locations are stored in `maps[(case, sensor)]['outside']`. They warn about extrapolation but do not measure uncertainty or guarantee accuracy elsewhere. The bootstrap intervals apply to station error measures only.


In [ ]:
# B2-B4: reuse the A2 layout, unchanged SPM values and colour scale
RESULT_OUT = PROJECT/'figures/final_results'
RESULT_OUT.mkdir(parents=True, exist_ok=True)
MAP_VIEWS = {'Belgium': (495,525,5685,5712), 'Guiana': (345,390,535,570)}  # UTM km
map_labels = {(s.case,s.sensor): f'{s.sensor} · {s.date} · {s.pixel_size_m} m'
              for s in scenes.itertuples()}

def plot_map_pair(keys, name, show=True):
    if not all(key in maps for key in keys):
        print('Waiting for regional rasters:', [key for key in keys if key not in maps])
        return
    panels = []
    for key in keys:
        scene = scenes.loc[scenes['case'].eq(key[0]) & scenes['sensor'].eq(key[1])].iloc[0]
        title = f'Sentinel-{scene.sensor[-1]} · {scene.date}'
        panels.append((key,maps[key]['spm'],title,(0,100),'viridis','Estimated SPM (mg/L)'))
    return a2_plot(name, panels, MAP_VIEWS[scene.region], show=show,
                   images=maps, labels=map_labels, out=RESULT_OUT)


In [ ]:
# B2: Belgian coastal detail on two dates
belgium_time = plot_map_pair([('Belgium_reference', 'S3'), ('Belgium_second', 'S3')],
                            'belgium_time')


In [ ]:
# B4: same close-up, acquisitions about 57 minutes apart
belgium_sensors = plot_map_pair([('Belgium_reference', 'S2'), ('Belgium_reference', 'S3')],
                               'belgium_sensors')


In [ ]:
# B3-B4: Guiana coastal detail, unchanged Belgian models
guiana_sensors = plot_map_pair([('Guiana_external', 'S2'), ('Guiana_external', 'S3')],
                              'guiana_sensors')
for scene in scenes.itertuples():
    key = scene.case, scene.sensor
    plt.close(plot_map_pair([key], 'panel_'+scene.case+'_'+scene.sensor, show=False))


Both regions show areas with higher estimated SPM near the coast than offshore. The Belgian dates and paired sensors show different patterns and quality gaps. Two dates cannot establish a seasonal or long-term trend, and sensor differences also include acquisition time and processing.

In the full Guiana rasters, 3.14% of valid S2 pixels and 1.63% of S3 pixels are outside the training ratio range. The Belgian S2 fraction is only 0.07%, yet includes an implausible 78,361 mg/L prediction. We flag these values and include them in the results. The weak station-test scores and extreme pixel predictions limit quantitative use. Guiana map accuracy has not been measured independently. [Prediction checks](mapping_01/map_checks.json).


### Optical comparison (A2, B2–B4)

The optical image above each SPM map shows the reflectance used to create it. Red-edge, red and green are assigned to RGB: S2 B5/B4/B3 and S3 Oa11/Oa08/Oa06, about 704 nm (S2) or 709 nm (S3), 665 nm and 560 nm. Each pair uses the same acquisition, pixel grid, quality mask and extent.

Red and red-edge sensitivity to turbidity motivated this view ([Chowdhury et al., 2023, §2.4 and §4](https://doi.org/10.3389/fmars.2023.1186441)). We chose the RGB assignment using the [false-colour principle](https://eo.belspo.be/en/iv22-true-and-false-colour-composites). For display, reflectance is divided by 0.03 sr⁻¹, limited to 0–1, and square-root stretched to show weaker signals. This does not change the reflectance data. Other constituents and processing effects can also affect colour. Because the optical images share the model's red and green inputs, their agreement is not independent validation.


In [ ]:
# A2-B4: same template and exact close-up as the SPM comparison
def compare_optical(keys, name, reference):
    optical, estimates = [], []
    for key in keys:
        data = maps[key]
        rgb = np.stack([data['red_edge'], data['red'], data['green']], axis=-1)
        rgba = np.dstack([np.nan_to_num(np.sqrt(np.clip(rgb/.03,0,1))),data['valid']])
        scene = scenes.loc[scenes['case'].eq(key[0]) & scenes['sensor'].eq(key[1])].iloc[0]
        label = f'{scene.sensor} · {scene.date}'
        optical.append((key,rgba,label+' · false colour',None,None,''))
        estimates.append((key,data['spm'],label+' · SPM',(0,100),'viridis','Estimated SPM (mg/L)'))
    view = (*reference.axes[0].get_xlim(), *reference.axes[0].get_ylim())
    return a2_plot(name, optical+estimates, view, images=maps,
                   labels=map_labels, out=RESULT_OUT, columns=2)


In [ ]:
# B2: optical patterns above the same SPM close-ups
optical_belgium_time = compare_optical([('Belgium_reference', 'S3'), ('Belgium_second', 'S3')],
    'belgium_time_optical', belgium_time)


In [ ]:
# B4: matched views, acquisitions about 57 minutes apart
optical_belgium_sensors = compare_optical([('Belgium_reference', 'S2'), ('Belgium_reference', 'S3')],
    'belgium_sensors_optical', belgium_sensors)


In [ ]:
# B3-B4: matched Guiana views, acquisitions about 50 minutes apart
optical_guiana = compare_optical([('Guiana_external', 'S2'), ('Guiana_external', 'S3')],
    'guiana_sensors_optical', guiana_sensors)
